# A1 Mini Camera Test Client

This notebook allows you to test the A1 Mini camera by sending a capture command via MQTT and receiving the image URI.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AccelerationConsortium/ac-training-lab/blob/main/src/ac_training_lab/a1_cam/test_camera.ipynb)

## Setup

First, install the required packages:

In [ ]:
!pip install paho-mqtt requests pillow

## Configure Credentials

This notebook uses [Colab Secrets](https://x.com/GoogleColab/status/1719798406195867814) to securely store credentials.

**To set up secrets in Colab:**
1. Click the key icon 🔑 in the left sidebar
2. Add the following secrets:
   - `MQTT_HOST` - Your MQTT broker host (e.g., `xxxxx.s1.eu.hivemq.cloud`)
   - `MQTT_USERNAME` - Your MQTT username
   - `MQTT_PASSWORD` - Your MQTT password
   - `DEVICE_SERIAL` - Your device serial number
3. Enable "Notebook access" for each secret

These values should match your `my_secrets.py` on the Raspberry Pi.

In [ ]:
from google.colab import userdata

# Load credentials from Colab secrets
MQTT_HOST = userdata.get('MQTT_HOST')
MQTT_PORT = 8883  # TLS port (use 1883 for non-TLS)
MQTT_USERNAME = userdata.get('MQTT_USERNAME')
MQTT_PASSWORD = userdata.get('MQTT_PASSWORD')
DEVICE_SERIAL = userdata.get('DEVICE_SERIAL')

# Topics (automatically generated from DEVICE_SERIAL)
CAMERA_READ_TOPIC = f"bambu_a1_mini/request/{DEVICE_SERIAL}"
CAMERA_WRITE_TOPIC = f"bambu_a1_mini/response/{DEVICE_SERIAL}"

print(f"Configured for device: {DEVICE_SERIAL}")
print(f"MQTT Host: {MQTT_HOST}")

## MQTT Client Functions

These functions handle the MQTT connection and message exchange:

In [ ]:
import json
from queue import Queue, Empty
import paho.mqtt.client as mqtt

data_queue = Queue()

def get_paho_client(sensor_data_topic, hostname, username, password=None, port=8883, tls=True):
    """Create and configure MQTT client"""
    print(f"Creating MQTT client for {hostname}:{port}")
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, protocol=mqtt.MQTTv5)
    
    def on_message(client, userdata, msg):
        print(f"✓ Message received on topic: {msg.topic}")
        print(f"  Payload: {msg.payload}")
        try:
            data = json.loads(msg.payload)
            print(f"  Parsed data: {data}")
            data_queue.put(data)
            print(f"  Data added to queue (queue size: {data_queue.qsize()})")
        except json.JSONDecodeError as e:
            print(f"  ✗ JSON decode error: {e}")
    
    def on_connect(client, userdata, flags, rc, properties=None):
        if rc == 0:
            print(f"✓ Connected successfully to MQTT broker")
        else:
            print(f"✗ Connection failed with result code {rc}")
        client.subscribe(sensor_data_topic, qos=2)
        print(f"✓ Subscribed to topic: {sensor_data_topic}")
    
    def on_disconnect(client, userdata, rc, properties=None):
        print(f"Disconnected with result code {rc}")
    
    client.on_connect = on_connect
    client.on_message = on_message
    client.on_disconnect = on_disconnect
    
    if tls:
        client.tls_set(tls_version=mqtt.ssl.PROTOCOL_TLS_CLIENT)
        print("TLS enabled")
    
    client.username_pw_set(username, password)
    print(f"Connecting to {hostname}:{port}...")
    client.connect(hostname, port)
    client.subscribe(sensor_data_topic, qos=2)
    
    return client

def send_and_receive(client, command_topic, msg, queue_timeout=60):
    """Send command and wait for response"""
    payload = json.dumps(msg)
    print(f"Publishing to topic: {command_topic}")
    print(f"  Payload: {payload}")
    result = client.publish(command_topic, payload, qos=2)
    print(f"  Publish result: {result.rc} (0=success)")
    
    client.loop_start()
    print("MQTT loop started")
    
    print(f"Waiting for response (timeout: {queue_timeout}s)...")
    try:
        data = data_queue.get(True, queue_timeout)
        print(f"✓ Response received")
        client.loop_stop()
        return data
    except Empty:
        client.loop_stop()
        print(f"✗ Timeout after {queue_timeout}s - no response received")
        print("  Possible issues:")
        print("  - Device is not running or not connected")
        print("  - Incorrect DEVICE_SERIAL in secrets")
        print("  - MQTT topics don't match device configuration")
        print(f"  - Device is listening on: {command_topic.replace('/request', '/response')}")
        raise TimeoutError(f"No response received after {queue_timeout}s")

## Capture and Display Image

Send a capture command and display the resulting image:

In [ ]:
import requests
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt

# Create MQTT client
print("="*60)
print("MQTT Camera Test")
print("="*60)
print(f"Connecting to MQTT broker at {MQTT_HOST}:{MQTT_PORT}...")
print(f"Device Serial: {DEVICE_SERIAL}")
print(f"Request Topic: {CAMERA_READ_TOPIC}")
print(f"Response Topic: {CAMERA_WRITE_TOPIC}")
print()

client = get_paho_client(
    CAMERA_WRITE_TOPIC, 
    MQTT_HOST, 
    MQTT_USERNAME, 
    password=MQTT_PASSWORD, 
    port=MQTT_PORT
)

# Send capture command
msg = {"command": "capture_image"}

try:
    print("\nSending capture command...")
    data = send_and_receive(client, CAMERA_READ_TOPIC, msg, queue_timeout=30)
    
    if "error" in data:
        print(f"\n✗ Error from device: {data['error']}")
    else:
        image_uri = data["image_uri"]
        print(f"\n✓ Image URI received: {image_uri}")
        
        # Download and display image
        print("Downloading image...")
        response = requests.get(image_uri)
        response.raise_for_status()
        print(f"✓ Image downloaded ({len(response.content)} bytes)")
        
        img = Image.open(BytesIO(response.content))
        
        # Display in notebook
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.title('Captured Image from A1 Mini Camera')
        plt.show()
        
        print(f"\n✓ Image size: {img.size}")
        print("✓ Image captured and displayed successfully!")
        
except TimeoutError as e:
    print(f"\n✗ {e}")
except Exception as e:
    print(f"\n✗ Unexpected error: {e}")
    import traceback
    traceback.print_exc()
finally:
    client.loop_stop()
    client.disconnect()
    print("\nDisconnected from MQTT broker")

## Troubleshooting

If you encounter issues:

1. **Connection timeout**: Verify your MQTT credentials and that the device is running
2. **Image download fails**: Check that your S3 bucket permissions allow public access or that you have proper IAM credentials
3. **Device not responding**: Check the device logs with `sudo journalctl -u a1-cam.service -f`

For more information, see the [A1 Mini Camera documentation](https://github.com/AccelerationConsortium/ac-training-lab/tree/main/src/ac_training_lab/a1_cam).